# Composite scores

Two parts, kept apart on purpose:

1. **Reported results** -- read straight from the pipeline outputs (`run_composite.py`,
   `run_ranking_rf.py`). These are the leakage-safe out-of-fold numbers that go in the thesis.
2. **Exploration** -- an in-sample composite built here from a different metric selection.
   Useful for poking at the construction; **not** what is reported.


In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.datasets import cfg
from config.paths import RAW_DATA_DIR, EVALUATION_DIR, METRIC_YAML
from preprocessing.metric_meta import load_window_families, load_quality
from preprocessing.align import load_metric_directions
from analysis.distributions import get_numeric_metrics
from analysis.evaluation import (calculate_all_metrics, load_metric_families, get_family,
                                 plot_metric_scores, plot_spearman_heatmap)
from analysis.composite import select_decorrelated_metrics, build_composites, quality_filter

# Uniform figure styling: one validated palette and recessive axes/grid for every figure
# in the project (config/palette.py). Call once, before plotting.
from config.palette import apply_plot_style
apply_plot_style()

base = RAW_DATA_DIR / cfg.name              # base tables
out = EVALUATION_DIR / cfg.name             # pipeline outputs


## 1. Reported results (pipeline outputs)

`composite/composite_cv_eval.csv` and the two `cv_by_fold.csv` files. All out-of-fold on
lineage-grouped folds, and the composite and RF stages use **identical folds**, so the two are
directly comparable.

**PR-AUC is the primary measure** (with precision@10%); ROC-AUC is secondary.


In [ ]:
summary_cv = pd.read_csv(out / "composite" / "composite_cv_eval.csv")
rf_cv = pd.read_csv(out / "ranking" / "rf_cv_metrics.csv").set_index("metric")

# the reported head-to-head, primary measures first
cols = ["model", "pr_auc", "pr_auc_fold_mean", "pr_auc_fold_std", "precision_at_pct", "auroc", "n_metrics"]
head_to_head = summary_cv[cols].copy()
head_to_head.loc[len(head_to_head)] = {
    "model": "rf",
    "pr_auc": rf_cv.loc["pr_auc_oof", "value"],
    "pr_auc_fold_mean": rf_cv.loc["pr_auc_fold_mean", "value"],
    "pr_auc_fold_std": rf_cv.loc["pr_auc_fold_mean", "std"],
    "precision_at_pct": rf_cv.loc["precision_at_10pct", "value"],
    "auroc": rf_cv.loc["roc_auc_oof", "value"],
    "n_metrics": np.nan,
}
display(head_to_head.round(4))
print("NOTE: 'best_single' is IN-SAMPLE and was selected over 46 metrics -- not comparable"
      " with the out-of-fold rows above it.")


In [ ]:
raw = pd.read_csv(base / "merged.csv")
scaled_aligned = pd.read_csv(base / "merged_scaled_aligned.csv")

# Evaluate on LABELLED rows only (designs have binder == "?"), and drop window families
# (net_charge, surface_hydrophobicity, interface_dSASA) from monotonic scoring -- they are
# two-sided, so a single-direction ranking is undefined. Consistent with run_evaluation.
families = load_metric_families(METRIC_YAML)
directions = load_metric_directions(METRIC_YAML)
lab = pd.to_numeric(raw["binder"], errors="coerce").isin([0, 1]).to_numpy()
metrics = [m for m in get_numeric_metrics(raw) if get_family(m, families) not in load_window_families()]
prevalence = float(pd.to_numeric(raw.loc[lab, "binder"]).mean())

rankings = calculate_all_metrics(raw[lab], metrics, directions)
print(f"{lab.sum()} labelled rows, {len(metrics)} metrics, prevalence {prevalence:.3f}")
rankings.head(10)


## 2. Exploration -- in-sample composite build

**Not the reported result.** This builds a composite from a *different* metric selection
(`select_decorrelated_metrics`, top 20 by family) than the pipeline uses
(`stability_select`, 11 metrics), and scores it on the same rows it was weighted on.
Both differences make the numbers optimistic. Compare against section 1, never instead of it.


In [ ]:
combine_cols = select_decorrelated_metrics(rankings, scaled_aligned[lab], top_n=20, method="family")
print("combining:", combine_cols)

composed = build_composites(scaled_aligned[lab], combine_cols, rankings=rankings)
composite_cols = [c for c in composed.columns if c.startswith("composite_")]

# NOTE: this re-evaluates the composite on the SAME rows used to build/weight it (in-sample),
# so these composite numbers are OPTIMISTIC. The leakage-free estimate is produced by
# run_composite.py (out-of-fold CV -> composite_cv_eval.csv). Kept here as a quick visual only.
combined = calculate_all_metrics(composed, metrics + composite_cols, {})
combined["is_composite"] = combined["metric"].isin(composite_cols)
combined.head(20)[["metric", "pr_auc", "aligned_roc", "is_composite"]]

In [ ]:
# Where do the composites land against the best singles?
# Sorted dot plot, one panel per measure, each against its OWN baseline -- PR-AUC against the
# positive-class prevalence, ROC-AUC against 0.5. (F1 is not reported: it was dropped from the
# benchmark in favour of PR-AUC + precision@10%.)
print(combined[combined["is_composite"]][["metric", "pr_auc", "aligned_roc"]].to_string(index=False))

combined["prevalence"] = prevalence          # plot_metric_scores draws the per-measure baseline
plot_metric_scores(combined, top_n=15, sort_by="pr_auc", show=True)


## Filter -> re-rank

In [ ]:
quality_cutoffs, quality_mode = load_quality()
flagged = quality_filter(raw[lab], quality_cutoffs, directions, mode=quality_mode)
print(flagged["quality_flag"].value_counts().to_dict())
kept = flagged[flagged["quality_flag"] == "ok"]

filtered_rankings = calculate_all_metrics(kept, metrics, directions)

before = rankings[["metric", "pr_auc"]].rename(columns={"pr_auc": "pr_auc_all"})
after = filtered_rankings[["metric", "pr_auc"]].rename(columns={"pr_auc": "pr_auc_filtered"})
compare = before.merge(after, on="metric").sort_values("pr_auc_all", ascending=False)
compare["delta"] = (compare["pr_auc_filtered"] - compare["pr_auc_all"]).round(4)
compare.head(15)

## Model-agreement diagnostics

Folded in from the exploratory tooling: how much do the models agree, and does constraining / templating change the scores?

In [ ]:
# Model agreement on iPTM. Correlation is POLARITY, so this uses the project's diverging map
# with a neutral grey midpoint and symmetric limits -- not a red-yellow-green ramp, which is a
# rainbow, is not colourblind-safe, and puts a hue at "no relationship".
# Restricted to the MODEL-level iPTM columns: a bare "endswith _iptm" also catches
# *_per_chain_pair_iptm, which is a per-chain-pair array summary, not a model.
from config.analysis import MODELS

iptm_cols = [f"{m}_iptm" for m in MODELS if f"{m}_iptm" in raw.columns]
if iptm_cols:
    corr = raw[iptm_cols].corr(method="spearman")
    corr.index = corr.columns = [c.replace("_iptm", "") for c in iptm_cols]
    plot_spearman_heatmap(corr, show=True)
